# Los tres tipos de métodos

Hasta ahora, todos los métodos que hemos escrito reciben `self` como primer
parámetro y operan sobre **un objeto concreto** (depositar dinero en *esta* cuenta,
saludar a *este* usuario...). Son los **métodos de instancia**, y son los más
habituales.

Pero Python ofrece **tres** tipos de métodos, y elegir bien entre ellos hace que el
código quede mucho más claro. En este vídeo veremos:

- Los **métodos de instancia** (`self`).
- Los **métodos de clase** (`@classmethod`, reciben `cls`).
- Los **métodos estáticos** (`@staticmethod`, no reciben ni `self` ni `cls`).
- Una tabla de decisión para saber cuál usar en cada caso.

Usaremos como ejemplo una clase `Cuenta` (una cuenta bancaria sencilla) a lo largo de
todo el vídeo.

---

## 1. Métodos de instancia: operar sobre *un* objeto

Es el método de siempre. Recibe `self` (el propio objeto) y trabaja con su **estado**.
Tiene sentido cuando la acción depende de los datos de una instancia concreta.

In [ ]:
class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo

    def depositar(self, cantidad):  # método de instancia: usa self.saldo
        self.saldo += cantidad

    def saldo_actual(self):
        return f"{self.titular} tiene {self.saldo}€"


c = Cuenta("Ana", 100)
c.depositar(50)
print(c.saldo_actual())  # Ana tiene 150€

Nada nuevo aquí: `depositar` modifica el saldo de **esta** cuenta (`c`), no de otra.
Sin una instancia, el método no tiene sentido: ¿depositar en qué cuenta?

---

## 2. Métodos de clase: operar sobre *la clase*

A veces queremos un método que no actúe sobre un objeto concreto, sino sobre la
**clase entera**. Para eso usamos el decorador `@classmethod`, y su primer parámetro
ya no es `self` sino **`cls`** (la propia clase).

Un ejemplo típico: un dato compartido por todas las cuentas, como el tipo de interés,
que queremos poder cambiar de golpe para todas.

In [ ]:
class Cuenta:
    tipo_interes = 0.02  # atributo de clase, compartido por todas las cuentas

    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo

    @classmethod
    def cambiar_interes(cls, nuevo):  # recibe la CLASE (cls), no una instancia
        cls.tipo_interes = nuevo


print("Interés inicial:", Cuenta.tipo_interes)

Cuenta.cambiar_interes(0.05)
print("Interés nuevo:", Cuenta.tipo_interes)

Fíjate en que llamamos al método sobre la **clase** (`Cuenta.cambiar_interes(...)`),
sin necesidad de crear ninguna cuenta. El cambio afecta a **todas** las instancias,
porque `tipo_interes` es un atributo de clase.

> 💡 `self` es *este* objeto; `cls` es *la clase*. Son solo nombres por convención
> (el primer parámetro), pero respétalos: cualquier persona que lea tu código espera
> ver `self` en un método de instancia y `cls` en uno de clase.

---

## 3. Métodos estáticos: funciones que viven en la clase

El tercer tipo es el **método estático** (`@staticmethod`). No recibe `self` **ni**
`cls`: no necesita ni la instancia ni la clase. Es, simplemente, una función normal
que colocamos **dentro** de la clase porque conceptualmente pertenece a ella.

Por ejemplo, una utilidad para comprobar si una cantidad es válida: no depende de
ninguna cuenta concreta, pero tiene sentido tenerla junto a `Cuenta`.

In [ ]:
class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo

    @staticmethod
    def es_cantidad_valida(cantidad):  # ni self ni cls
        return cantidad > 0

    def depositar(self, cantidad):
        # podemos reutilizar la utilidad desde un método de instancia
        if not Cuenta.es_cantidad_valida(cantidad):
            raise ValueError("La cantidad debe ser positiva")
        self.saldo += cantidad


print(Cuenta.es_cantidad_valida(50))   # True
print(Cuenta.es_cantidad_valida(-10))  # False

c = Cuenta("Ana", 100)
c.depositar(50)
print(c.saldo)  # 150

In [ ]:
# Si intentamos depositar una cantidad inválida, falla de forma controlada
try:
    c.depositar(-30)
except ValueError as e:
    print("Error:", e)

> 💡 ¿Y por qué no dejarlo como una función suelta fuera de la clase? Podrías, pero
> meterla como `@staticmethod` la mantiene **agrupada** con la clase a la que
> pertenece. Es una cuestión de organización y legibilidad.

---

## 4. Los tres juntos

Veámoslos conviviendo en la misma clase, para tener la foto completa:

In [ ]:
class Cuenta:
    tipo_interes = 0.02

    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo

    # 1) Instancia: opera sobre ESTA cuenta
    def depositar(self, cantidad):
        self.saldo += cantidad

    # 2) Clase: opera sobre la CLASE (cambia el interés de todas)
    @classmethod
    def cambiar_interes(cls, nuevo):
        cls.tipo_interes = nuevo

    # 3) Estático: utilidad relacionada, sin self ni cls
    @staticmethod
    def es_cantidad_valida(cantidad):
        return cantidad > 0


c = Cuenta("Ana", 100)
c.depositar(50)
Cuenta.cambiar_interes(0.05)

print(c.saldo)                          # 150  (instancia)
print(Cuenta.tipo_interes)              # 0.05 (clase)
print(Cuenta.es_cantidad_valida(-10))   # False (estático)

---

## 5. Tabla de decisión: ¿cuál uso?

La pregunta clave es: **¿qué necesita el método para funcionar?**

| Tipo | 1.er parámetro | ¿Usa la instancia? | ¿Usa la clase? | Cuándo usarlo |
|------|----------------|--------------------|----------------|----------------|
| **Instancia** | `self` | Sí | — | Operar sobre el estado de un objeto concreto |
| **Clase** (`@classmethod`) | `cls` | No | Sí | Tocar estado de clase o crear objetos de formas alternativas |
| **Estático** (`@staticmethod`) | (ninguno) | No | No | Función auxiliar que vive en la clase por organización |

Una forma rápida de decidir:

- ¿Necesitas los datos de **un objeto** (`self.algo`)? → **método de instancia**.
- ¿Necesitas la **clase** (`cls`) pero no una instancia? → **`@classmethod`**.
- ¿No necesitas **ninguno** de los dos? → **`@staticmethod`**.

---

## 6. Resumen

En este vídeo hemos visto que Python tiene **tres tipos de métodos**:

1. **Métodos de instancia**: reciben `self` y trabajan con el estado de un objeto
   concreto. Son los más comunes.
2. **Métodos de clase** (`@classmethod`): reciben `cls` y operan sobre la propia
   clase (por ejemplo, modificar un atributo de clase compartido).
3. **Métodos estáticos** (`@staticmethod`): no reciben ni `self` ni `cls`; son
   funciones auxiliares que agrupamos dentro de la clase por organización.

La regla para elegir es preguntarte **qué necesita el método**: la instancia, la
clase, o ninguna de las dos.

| Tipo | Decorador | Recibe |
|------|-----------|--------|
| Instancia | (ninguno) | `self` |
| Clase | `@classmethod` | `cls` |
| Estático | `@staticmethod` | nada |